# TechMind — Limpieza de texto

### Equipo tejONEs

Este notebook continúa el flujo de `05_exploracion_dataset_final.ipynb` y aplica al dataset unificado la misma función que utilizará Backend. El proceso decodifica entidades HTML, elimina etiquetas y contenido no visible de `script` y `style`, convierte el texto a minúsculas, reemplaza puntuación Unicode, normaliza espacios y elimina stopwords en español de NLTK.

La implementación canónica vive en `shared/limpieza_texto.py`; no se redefine aquí.

## Flujo, preparación y reutilización

Los notebooks se ejecutan en este orden:

1. `05_exploracion_dataset_final.ipynb`: consolida y genera el dataset unificado de entrada.
2. `06_limpieza_texto.ipynb`: aplica la limpieza y genera la columna `texto_limpio`.

Después de instalar las dependencias, el corpus de stopwords se prepara una sola vez por entorno:

```bash
python -m nltk.downloader stopwords
```

Data Science y Backend deben importar la misma función para procesar de forma idéntica los textos de entrenamiento y los textos nuevos. La función acepta tanto texto plano como contenido con etiquetas o entidades HTML:

```python
from shared.limpieza_texto import limpiar_texto

texto_limpio = limpiar_texto("&lt;p&gt;¡Curso práctico de Python para Backend!&lt;/p&gt;")
```

## 1. Importaciones y rutas

In [ ]:
from pathlib import Path
import sys
import time

import nltk
import pandas as pd

In [2]:
def find_project_root() -> Path:
    
    for candidata in (Path.cwd(), *Path.cwd().parents):
        if (candidata / 'shared' / 'limpieza_texto.py').is_file():
            return candidata
    raise FileNotFoundError('No se encontró la raíz del repositorio.')


RAIZ_REPOSITORIO = find_project_root()
if str(RAIZ_REPOSITORIO) not in sys.path:
    sys.path.insert(0, str(RAIZ_REPOSITORIO))

# Importación de la funcion limpiar_texto del script limpieza_texto.py
# Este script se encuentra en data_science/shared/
from shared.limpieza_texto import limpiar_texto


RUTA_ENTRADA = (
    RAIZ_REPOSITORIO
    / 'data_science'
    / 'data'
    / 'procesados'
    / 'dataset_FINAL_UNIFICADO_techmind.csv'
)
RUTA_SALIDA = RUTA_ENTRADA.with_name(
    'dataset_FINAL_UNIFICADO_techmind_limpio.csv'
)

print(f'Entrada: {RUTA_ENTRADA.relative_to(RAIZ_REPOSITORIO)}')
print(f'Salida: {RUTA_SALIDA.relative_to(RAIZ_REPOSITORIO)}')

Entrada: data_science\data\procesados\dataset_FINAL_UNIFICADO_techmind.csv
Salida: data_science\data\procesados\dataset_FINAL_UNIFICADO_techmind_limpio.csv


## 2. Verificación de recursos y carga del dataset

In [3]:
try:
    nltk.data.find('corpora/stopwords')
except LookupError as error:
    raise RuntimeError(
        'Falta el corpus de stopwords. Ejecuta: '
        'python -m nltk.downloader stopwords'
    ) from error

if not RUTA_ENTRADA.is_file():
    raise FileNotFoundError(f'No existe el dataset de entrada: {RUTA_ENTRADA}')

df = pd.read_csv(RUTA_ENTRADA)
if 'texto' not in df.columns:
    raise ValueError("El dataset no contiene la columna requerida 'texto'.")

print(f'Registros cargados: {len(df):,}')
print(f'Columnas originales: {list(df.columns)}')

Registros cargados: 1,400
Columnas originales: ['titulo', 'texto', 'categoria', 'autor', 'tipo']


## 3. Validación del tratamiento de HTML

Antes de procesar el dataset completo se comprueban entidades con nombre, entidades numéricas, etiquetas y contenido no visible. Esto evita introducir tokens artificiales como `lt`, `gt` o `nbsp` en el corpus.

In [4]:
casos_html = {
    '&lt;p&gt;Resumen&nbsp;de Python&lt;/p&gt;': 'resumen python',
    'Programaci&#243;n para Backend': 'programación backend',
    '<style>.oculto { color: red; }</style><p>Curso de Python</p><script>alert(1)</script>': 'curso python',
}

resultados_html = []
for texto_original, resultado_esperado in casos_html.items():
    resultado_obtenido = limpiar_texto(
        texto_original,
        palabras_vacias={'de', 'para'},
    )
    assert resultado_obtenido == resultado_esperado
    resultados_html.append({
        'texto_original': texto_original,
        'texto_limpio': resultado_obtenido,
    })

pd.DataFrame(resultados_html)

,texto_original,texto_limpio
0,&lt;p&gt;Resumen&nbsp;de Python&lt;/p&gt;,resumen python
1,Programaci&#243;n para Backend,programación backend
2,<style>.oculto { color: red; }</style><p>Curso...,curso python


## 4. Aplicación de `limpiar_texto()` y exportación

In [5]:
inicio = time.perf_counter()
df['texto_limpio'] = df['texto'].fillna('').map(limpiar_texto)
duracion = time.perf_counter() - inicio

df.to_csv(RUTA_SALIDA, index=False, encoding='utf-8')

print(f'Registros procesados: {len(df):,}')
print(f'Tiempo de limpieza: {duracion:.2f} segundos')
print(
    'Dataset generado: '
    f'{RUTA_SALIDA.relative_to(RAIZ_REPOSITORIO)}'
)

Registros procesados: 1,400
Tiempo de limpieza: 0.23 segundos
Dataset generado: data_science\data\procesados\dataset_FINAL_UNIFICADO_techmind_limpio.csv


## 5. Validación del resultado

In [6]:
df_verificacion = pd.read_csv(RUTA_SALIDA)

assert len(df_verificacion) == len(df), 'Cambió el número de registros.'
assert 'texto_limpio' in df_verificacion.columns, 'No se generó texto_limpio.'
assert df_verificacion['texto_limpio'].notna().all(), 'Hay valores nulos en texto_limpio.'

resumen = pd.DataFrame({
    'metrica': [
        'registros',
        'textos originales vacíos',
        'textos limpios vacíos',
        'longitud media original',
        'longitud media limpia',
    ],
    'valor': [
        len(df_verificacion),
        int(df['texto'].fillna('').str.strip().eq('').sum()),
        int(df_verificacion['texto_limpio'].fillna('').str.strip().eq('').sum()),
        round(df['texto'].fillna('').str.len().mean(), 2),
        round(df_verificacion['texto_limpio'].fillna('').str.len().mean(), 2),
    ],
})
resumen

,metrica,valor
0,registros,1400.00
1,textos originales vacíos,0.00
2,textos limpios vacíos,0.00
3,longitud media original,1516.62
4,longitud media limpia,1097.83


In [7]:
pd.set_option('display.max_colwidth', 140)
df_verificacion[['texto', 'texto_limpio']].sample(10)


,texto,texto_limpio
1184,"Recientemente, ha surgido el concepto de realidad cruzada (XR) basado en la web que abarca la tecnología de realidad virtual (VR), reali...",recientemente surgido concepto realidad cruzada xr basado web abarca tecnología realidad virtual vr realidad aumentada ar realidad mixta...
385,"Aprenda a usar GitHub Copilot y Fabric Copilot para el desarrollo de bases de datos asistidas por IA en plataformas de Microsoft SQL, co...",aprenda usar github copilot fabric copilot desarrollo bases datos asistidas ia plataformas microsoft sql sql server azure sql microsoft ...
1199,Cierre Chrome (o Chromium) y reinicie con el argumento --disable-web-security. Acabo de probar esto y verifiqué que puedo acceder al con...,cierre chrome chromium reinicie argumento disable web security acabo probar verifiqué puedo acceder contenido iframe src= incrustado pág...
1014,Le invitamos a seminarios web gratuitos deTeams para la educación. En este webinar en particular veremos las evaluaciones de tipo cues...,invitamos seminarios web gratuitos deteams educación webinar particular veremos evaluaciones tipo cuestionario teams diseño asignación p...
71,"Iterator.remove() es seguro, puedes usarlo así: \n\n Lista&lt;String&gt; lista = nueva ArrayList&lt;&gt;();\n\n// Esta es una forma int...",iterator remove seguro puedes usarlo así lista lista = nueva arraylist < > forma inteligente crear iterador llamar iterator hasnext harí...
1044,Hay varias formas de comprobar si una variable es una matriz o no. La mejor solución es la que has elegido. \n variable.constructor ===...,varias formas comprobar si variable matriz mejor solución elegido variable constructor === matriz método rápido chrome probablemente dem...
309,"Este artículo es una retrospectiva de ocho años sobre las prioridades de desarrollo de RocksDB, una tienda de valores clave desarrollada...",artículo retrospectiva ocho años prioridades desarrollo rocksdb tienda valores clave desarrollada facebook apunta sistemas distribuidos ...
377,"No es una unión ya que la relación solo se evaluará cuando sea necesario. Por otro lado, una unión (en una base de datos SQL) resolverá ...",unión relación solo evaluará necesario lado unión base datos sql resolverá relaciones devolverá si sola tabla unirás dos tablas puede le...
1011,"Con flexbox es muy fácil diseñar el div centrado horizontal y verticalmente. \n \r\n \r\n #interior { \n borde: 0,05 em negro sólido...",flexbox fácil diseñar div centrado horizontal verticalmente interior borde 0 05 em negro sólido exterior borde 0 05 em rojo sólido ancho...
1031,"Aprenda conceptos fundamentales de programación (por ejemplo, funciones, bucles for, declaraciones condicionales) y cómo resolver proble...",aprenda conceptos fundamentales programación ejemplo funciones bucles for declaraciones condicionales cómo resolver problemas programado...
